In [5]:
# 로깅 설정
import logging
import asyncio
import aiohttp
import ssl
import certifi
import json
import pandas as pd
import time
import random
from urllib.request import urlopen
from datetime import datetime


# matplotlib.rc('font', family='Malgun Gothic')
# plt.rcParams['axes.unicode_minus'] = False
# warnings.filterwarnings("ignore")

#### 1. 상장기업 매출액 API 수신

In [6]:
# DB 접속 정보 설정
db_info = {
    'user': 'stox7412',         # 예: 'root'
    'password': 'Apt106503!~', # 예: '1234'
    # 'host': '192.168.0.230',
    'host': 'hystox74.synology.me',         # 예: 'localhost' 또는 IP
    'port': '3307',              # 기본 포트는 보통 3306
    'database': 'investar'        # 예: 'trade_data'
}

In [7]:
# def get_adata_annual(adata, data_name):
#     data = adata.pivot_table(index='date', columns='ticker', values=data_name).astype('float')
#     data = data.ffill(limit=11)  # Replace fillna(method='ffill') with ffill()
#     data = data.loc[[x for x in data.index if x.month == 6]].reindex(data.index).ffill(limit=11).dropna(how='all')
#
#     return data
#
# def get_last_quarter_end():
#     # 현재 날짜 가져오기
#     today = date.today()
#
#     # 분기 마지막 월 정의 (Q1: 3월, Q2: 6월, Q3: 9월, Q4: 12월)
#     quarter_months = [3, 6, 9, 12]
#
#     # 현재 분기의 시작 월 찾기
#     for m in quarter_months:
#         if today.month <= m:
#             last_quarter_month = quarter_months[quarter_months.index(m) - 1]
#             break
#     else:
#         last_quarter_month = 12  # 현재가 1월이라면, 작년 12월이 마지막 분기
#
#     # 연도 조정
#     year = today.year if last_quarter_month != 12 else today.year - 1
#
#     # 해당 월의 마지막 날짜 계산
#     last_day = calendar.monthrange(year, last_quarter_month)[1]
#
#     return date(year, last_quarter_month, last_day)
#
# def get_last_month_end():
#     # 현재 날짜 가져오기
#     today = date.today()
#
#     # 지난달 계산
#     last_month = today.month - 1 if today.month > 1 else 12
#     last_year = today.year if today.month > 1 else today.year - 1
#
#     # 지난달의 마지막 날 계산
#     last_day = calendar.monthrange(last_year, last_month)[1]
#
#     # 날짜 객체 생성
#     last_month_end = date(last_year, last_month, last_day)
#
#     # 문자열 변환 (YYYY-MM-DD 형식)
#     return last_month_end.strftime("%Y-%m-%d")

In [8]:
logging.basicConfig(level=logging.ERROR)
logger = logging.getLogger(__name__)

# API 키 설정
api_key = "hT0gAk87j9xZx4PlBApvBqfVL5IahvgV"

# 성공/실패 추적을 위한 전역 변수들
successful_tickers = []
failed_tickers = []
total_tickers = 0
processed_count = 0
start_time = None

# 더 보수적인 설정으로 변경
max_concurrent = 10      # 동시 요청 수를 크게 줄임
base_delay = 1.0        # 기본 딜레이 증가
retry_delay = 3.0       # 재시도 딜레이 증가
max_retries = 5         # 재시도 횟수 증가
batch_size = 100        # 배치 크기

def print_progress_bar(current, total, success_count, bar_length=40):
    """실시간 진행 상황 바 출력"""
    if total == 0:
        return

    progress = current / total
    filled_length = int(bar_length * progress)
    bar = '█' * filled_length + '░' * (bar_length - filled_length)

    # 시간 계산
    elapsed = time.time() - start_time if start_time else 0
    elapsed_str = f"{int(elapsed//60)}:{int(elapsed%60):02d}"

    if current > 0:
        avg_time_per_ticker = elapsed / current
        remaining_time = avg_time_per_ticker * (total - current)
        remaining_str = f"{int(remaining_time//60)}:{int(remaining_time%60):02d}"
    else:
        remaining_str = "계산중..."

    success_rate = (success_count / current * 100) if current > 0 else 0
    failed_count = current - success_count

    print(f"\r🔄 진행: [{bar}] {current}/{total} ({progress*100:.1f}%)")
    print(f"✅ 성공: {success_count} | ❌ 실패: {failed_count} | 📈 성공률: {success_rate:.1f}%")
    print(f"⏱️  경과: {elapsed_str} | 예상잔여: {remaining_str}")
    print("-" * 70)

def get_jsonparsed_data(url, timeout=15):
    """동기 방식으로 JSON 데이터 가져오기 (재시도 로직 포함)"""
    for attempt in range(3):
        try:
            context = ssl.create_default_context(cafile=certifi.where())
            with urlopen(url, context=context, timeout=timeout) as response:
                data = response.read().decode("utf-8")
                return json.loads(data)
        except Exception as e:
            print(f"⚠️  API 요청 실패 (시도 {attempt + 1}/3): {str(e)[:50]}...")
            if attempt < 2:
                time.sleep(2)
    return None

async def fetch_json_data_safe(url, session, semaphore, ticker, retry_count=None):
    """안전한 비동기 JSON 데이터 수집"""
    if retry_count is None:
        retry_count = max_retries

    async with semaphore:
        for attempt in range(retry_count):
            try:
                # 요청 간격을 랜덤하게 조절
                delay = base_delay + random.uniform(0.5, 2.0)
                if attempt > 0:  # 재시도인 경우 더 긴 대기
                    delay = retry_delay + random.uniform(1.0, 4.0)

                await asyncio.sleep(delay)

                print(f"📡 {ticker} 요청 중... (시도 {attempt + 1}/{retry_count})")

                async with session.get(url, timeout=20) as response:
                    if response.status == 200:
                        text_data = await response.text()
                        if text_data.strip():  # 빈 응답 체크
                            data = json.loads(text_data)
                            if data:  # 빈 데이터 체크
                                print(f"✅ {ticker} 성공!")
                                return data
                            else:
                                print(f"⚠️  {ticker} 빈 데이터 응답")
                        else:
                            print(f"⚠️  {ticker} 빈 응답")

                    elif response.status == 429:
                        # Rate limit 처리
                        wait_time = (2 ** attempt) + random.uniform(1, 5)
                        print(f"⏳ {ticker} Rate limit - {wait_time:.1f}초 대기...")
                        await asyncio.sleep(wait_time)

                    elif response.status == 403:
                        print(f"🚫 {ticker} API 권한 오류 (403)")
                        break  # 403은 재시도해도 의미없음

                    else:
                        print(f"❌ {ticker} HTTP {response.status}")

            except asyncio.TimeoutError:
                print(f"⏱️  {ticker} 타임아웃 (시도 {attempt + 1})")

            except json.JSONDecodeError:
                print(f"🔧 {ticker} JSON 파싱 오류 (시도 {attempt + 1})")

            except Exception as e:
                print(f"💥 {ticker} 오류: {str(e)[:30]}... (시도 {attempt + 1})")

            # 마지막 시도가 아니면 잠시 대기
            if attempt < retry_count - 1:
                extra_wait = random.uniform(2, 5)
                print(f"⏳ {ticker} {extra_wait:.1f}초 후 재시도...")
                await asyncio.sleep(extra_wait)

    print(f"💔 {ticker} 최종 실패")
    return None

async def fetch_stock_profile_safe(ticker, session, semaphore):
    """안전한 개별 주식 프로필 수집"""
    url = f"https://financialmodelingprep.com/api/v3/profile/{ticker}?apikey={api_key}"

    try:
        data = await fetch_json_data_safe(url, session, semaphore, ticker)

        if data and len(data) > 0:
            df = pd.DataFrame(data)
            successful_tickers.append(ticker)
            return df
        else:
            failed_tickers.append(ticker)
            return None

    except Exception as e:
        print(f"💥 {ticker} 예외 발생: {e}")
        failed_tickers.append(ticker)
        return None

async def download_profiles_batch(ticker_batch, batch_num, total_batches):
    """배치 단위로 프로필 다운로드"""
    print(f"\n🎯 배치 {batch_num}/{total_batches} 시작 ({len(ticker_batch)}개 종목)")
    print("=" * 60)

    # SSL 및 연결 설정 (더 보수적으로)
    ssl_context = ssl.create_default_context(cafile=certifi.where())
    connector = aiohttp.TCPConnector(
        ssl=ssl_context,
        limit=50,        # 연결 풀 크기 감소
        limit_per_host=20,  # 호스트당 연결 수 감소
        ttl_dns_cache=300,  # DNS 캐시 시간
        use_dns_cache=True
    )

    # 타임아웃 설정 증가
    timeout = aiohttp.ClientTimeout(
        total=60,      # 전체 타임아웃
        connect=15,    # 연결 타임아웃
        sock_read=20   # 소켓 읽기 타임아웃
    )

    semaphore = asyncio.Semaphore(max_concurrent)

    async with aiohttp.ClientSession(
        connector=connector,
        timeout=timeout,
        headers={'User-Agent': 'Mozilla/5.0 (compatible; DataCollector/1.0)'}
    ) as session:

        tasks = [
            fetch_stock_profile_safe(ticker, session, semaphore)
            for ticker in ticker_batch
        ]

        results = []
        completed = 0
        batch_start_success = len(successful_tickers)

        # 진행 상황을 실시간으로 모니터링
        for coro in asyncio.as_completed(tasks):
            try:
                result = await coro
                if result is not None:
                    results.append(result)

                completed += 1

                # 진행 상황 업데이트 (5개마다 또는 완료시)
                if completed % 5 == 0 or completed == len(tasks):
                    # 이전 출력 지우기
                    for _ in range(4):
                        print("\033[F\033[K", end="")

                    current_batch_success = len(successful_tickers) - batch_start_success
                    print_progress_bar(completed, len(tasks), current_batch_success)

            except Exception as e:
                completed += 1
                print(f"💥 태스크 실행 오류: {e}")

    batch_success = len(successful_tickers) - batch_start_success
    batch_failed = len(ticker_batch) - batch_success

    print(f"\n🏁 배치 {batch_num} 완료!")
    print(f"✅ 성공: {batch_success}개 | ❌ 실패: {batch_failed}개")
    print(f"📈 배치 성공률: {batch_success/len(ticker_batch)*100:.1f}%")

    return results

async def retry_failed_tickers(max_retry_attempts=2):
    """실패한 티커들 재시도"""
    global failed_tickers

    if not failed_tickers:
        print("🎉 재시도할 실패 티커가 없습니다!")
        return []

    print(f"\n🔄 실패한 {len(failed_tickers)}개 티커 재시도 시작...")

    # 재시도를 위해 실패 목록 복사
    retry_list = failed_tickers.copy()
    failed_tickers.clear()  # 실패 목록 초기화

    all_retry_results = []

    # 더 작은 배치로 재시도
    retry_batch_size = 50
    for attempt in range(max_retry_attempts):
        if not retry_list:
            break

        print(f"\n🔄 재시도 {attempt + 1}/{max_retry_attempts} 시작...")
        print(f"📊 재시도 대상: {len(retry_list)}개")

        current_retry_results = []

        # 작은 배치로 나누어 재시도
        for i in range(0, len(retry_list), retry_batch_size):
            batch = retry_list[i:i + retry_batch_size]
            batch_num = i // retry_batch_size + 1
            total_batches = (len(retry_list) + retry_batch_size - 1) // retry_batch_size

            print(f"\n🎯 재시도 배치 {batch_num}/{total_batches}")

            batch_results = await download_profiles_batch(
                batch, batch_num, total_batches
            )
            current_retry_results.extend(batch_results)

            # 배치 간 더 긴 휴식
            if i + retry_batch_size < len(retry_list):
                print("⏳ 재시도 배치 간 10초 휴식...")
                await asyncio.sleep(10)

        all_retry_results.extend(current_retry_results)

        # 다음 재시도를 위해 여전히 실패한 것들만 남김
        retry_list = failed_tickers.copy()
        failed_tickers.clear()

        if retry_list:
            print(f"⚠️  {len(retry_list)}개 종목이 여전히 실패상태")
            if attempt < max_retry_attempts - 1:
                print("⏳ 다음 재시도까지 15초 대기...")
                await asyncio.sleep(15)
        else:
            print("🎉 모든 재시도 성공!")
            break

    # 최종적으로 실패한 것들을 다시 실패 목록에 추가
    if retry_list:
        failed_tickers.extend(retry_list)

    return all_retry_results

# 이벤트 루프 감지 및 처리 함수
def get_or_create_eventloop():
    """현재 이벤트 루프가 실행 중인지 확인하고 적절하게 처리"""
    try:
        # 현재 실행 중인 이벤트 루프가 있는지 확인
        loop = asyncio.get_running_loop()
        return loop, True  # 실행 중인 루프가 있음
    except RuntimeError:
        # 실행 중인 루프가 없음
        return None, False

async def run_batch_processing():
    """전체 배치 처리를 담당하는 코루틴"""
    global all_results, total_batches

    all_results = []
    total_batches = (len(ticker_list) + batch_size - 1) // batch_size

    try:
        for i in range(0, len(ticker_list), batch_size):
            batch = ticker_list[i:i + batch_size]
            batch_num = i // batch_size + 1

            try:
                batch_results = await download_profiles_batch(batch, batch_num, total_batches)
                all_results.extend(batch_results)

            except Exception as e:
                print(f"💥 배치 {batch_num} 처리 중 오류: {e}")
                # 배치 전체가 실패한 경우 해당 티커들을 실패 목록에 추가
                failed_tickers.extend(batch)

            # 배치 간 휴식 (마지막 배치가 아닌 경우)
            if i + batch_size < len(ticker_list):
                print("\n⏳ 배치 간 5초 휴식...")
                await asyncio.sleep(5)

        print(f"\n🏁 1차 수집 완료!")
        print(f"✅ 성공: {len(successful_tickers)}개")
        print(f"❌ 실패: {len(failed_tickers)}개")

        # 실패한 티커들 재시도
        if failed_tickers:
            print(f"\n🔄 실패 티커들 재시도 시작...")
            retry_results = await retry_failed_tickers()
            all_results.extend(retry_results)

    except KeyboardInterrupt:
        print("\n⏹️  사용자에 의해 중단되었습니다.")
    except Exception as e:
        print(f"\n💥 예상치 못한 오류 발생: {e}")

# =================================================================
# 메인 실행 코드 시작
# =================================================================

print("🚀 " + "="*60)
print("📈 개선된 FMP API 주식 데이터 수집기 시작!")
print(f"⚙️  동시 요청 수: {max_concurrent}")
print(f"⏱️  기본 딜레이: {base_delay}초")
print(f"🔄 최대 재시도: {max_retries}회")
print(f"📦 배치 크기: {batch_size}")
print("="*60)

# 시작 시간 기록
start_time = time.time()

print("📋 주식 리스트 가져오는 중...")

# 1. 주식 리스트 가져오기
url = f"https://financialmodelingprep.com/api/v3/stock/list?apikey={api_key}"
stock_list = get_jsonparsed_data(url)

if not stock_list:
    print("❌ 주식 리스트를 가져올 수 없습니다.")
    exit()

print("✅ 주식 리스트 수신 완료!")

# DataFrame 변환 및 필터링
stock_listed = pd.DataFrame(stock_list)

print("📊 데이터 필터링 중...")

# 조건 설정
con1 = stock_listed['exchangeShortName'] == 'NASDAQ'
con2 = stock_listed['exchangeShortName'] == 'NYSE'
con3 = stock_listed['exchangeShortName'] == 'AMEX'
con4 = stock_listed['type'] == 'stock'

# 필터링
stock_screen1 = stock_listed[con1 | con2 | con3].copy()
con4_aligned = con4.reindex(stock_screen1.index)
stock_screen2 = stock_screen1[con4_aligned].copy()

us_stock_info = stock_screen2
ticker_list = us_stock_info['symbol'].unique().tolist()

print(f"📊 필터링 완료! 처리 대상 티커: {len(ticker_list)}개")

# 대용량 데이터 경고 및 테스트 모드 옵션
if len(ticker_list) > 1000:
    print(f"\n⚠️  대용량 데이터입니다. 전체 처리 시 수 시간이 소요될 수 있습니다.")
    print("📝 테스트용으로 처음 500개만 처리하려면 'y'를 입력하세요.")

    try:
        response = input("전체 처리하시겠습니까? (y: 테스트 500개, 그 외: 전체 처리): ")

        if response.lower() == 'y':
            ticker_list = ticker_list[:500]
            print(f"🎯 테스트 모드: {len(ticker_list)}개 티커만 처리합니다.")
        else:
            print(f"🎯 전체 모드: {len(ticker_list)}개 티커를 모두 처리합니다.")
    except:
        print(f"🎯 입력 없음. 전체 {len(ticker_list)}개 티커를 처리합니다.")

total_tickers = len(ticker_list)

print(f"\n🚀 데이터 수집 시작!")
print(f"📊 총 {total_tickers}개 종목 처리 예정")

# 2. 이벤트 루프 상태에 따른 실행 방식 결정
loop, is_running = get_or_create_eventloop()

if is_running:
    # 이미 실행 중인 이벤트 루프가 있는 경우 (Jupyter 등)
    print("🔄 기존 이벤트 루프에서 실행 중...")
    import nest_asyncio
    nest_asyncio.apply()

    # 태스크로 실행
    task = asyncio.create_task(run_batch_processing())
    await task
else:
    # 새로운 이벤트 루프 생성해서 실행
    print("🔄 새 이벤트 루프 생성해서 실행...")
    asyncio.run(run_batch_processing())

# 4. 결과 합치기
if all_results:
    info = pd.concat([df for df in all_results if df is not None], ignore_index=True)
    print(f"✅ 최종 수집된 데이터: {len(info)}개 프로필")
else:
    info = pd.DataFrame()
    print("❌ 수집된 데이터가 없습니다.")

# 5. 최종 리포트
total_time = time.time() - start_time if start_time else 0

print("\n" + "🏁" * 60)
print("🎊 데이터 수집 최종 완료!")
print("🏁" * 60)

print(f"📊 총 처리 대상: {total_tickers}개")
print(f"✅ 성공: {len(successful_tickers)}개")
print(f"❌ 최종 실패: {len(failed_tickers)}개")
print(f"📈 최종 성공률: {len(successful_tickers)/total_tickers*100:.1f}%")
print(f"⏱️  총 소요시간: {int(total_time//3600)}시간 {int((total_time%3600)//60)}분 {int(total_time%60)}초")
print(f"⚡ 평균 처리시간: {total_time/total_tickers:.2f}초/종목")

# 6. 중요 종목들 상태 체크
important_tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'META', 'NVDA', 'TEL']
print(f"\n🎯 주요 종목 수집 상태:")
for ticker in important_tickers:
    if ticker in successful_tickers:
        print(f"✅ {ticker}: 성공")
    elif ticker in failed_tickers:
        print(f"❌ {ticker}: 실패")
    elif ticker in ticker_list:
        print(f"❓ {ticker}: 처리 대상이었으나 결과 없음")
    else:
        print(f"➖ {ticker}: 처리 대상 아님")

# 7. 실패 티커 처리
if failed_tickers:
    print(f"\n💔 실패 티커 샘플 (처음 20개):")
    print(failed_tickers[:20])

    # 실패 티커를 파일로 저장
    failed_df = pd.DataFrame({
        'failed_ticker': failed_tickers,
        'timestamp': [datetime.now().strftime('%Y-%m-%d %H:%M:%S')] * len(failed_tickers)
    })
    failed_filename = f"failed_tickers_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    failed_df.to_csv(failed_filename, index=False)
    print(f"💾 실패 티커들이 {failed_filename}에 저장되었습니다.")

# 8. 성공 티커를 파일로 저장
if successful_tickers:
    successful_df = pd.DataFrame({
        'successful_ticker': successful_tickers,
        'timestamp': [datetime.now().strftime('%Y-%m-%d %H:%M:%S')] * len(successful_tickers)
    })
    success_filename = f"successful_tickers_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    successful_df.to_csv(success_filename, index=False)
    print(f"💾 성공 티커들이 {success_filename}에 저장되었습니다.")

print("🏁" * 60)

# 9. 수집 결과 미리보기
if not info.empty:
    print(f"\n📈 수집 결과 미리보기:")
    print(info.head())
    print(f"\n📊 데이터 형태: {info.shape}")
    print(f"📋 컬럼: {info.columns.tolist()}")

    if 'symbol' in info.columns:
        print(f"📋 수집된 종목 예시: {info['symbol'].head(10).tolist()}")

        # 최종 DataFrame에서 중요 티커들 확인
        print(f"\n🔍 최종 DataFrame에서 주요 종목 확인:")
        for ticker in important_tickers:
            ticker_data = info[info['symbol'] == ticker]
            if not ticker_data.empty:
                print(f"✅ {ticker}: DataFrame에 포함됨 ({len(ticker_data)} records)")
            else:
                print(f"❌ {ticker}: DataFrame에 없음")
else:
    print(f"\n❌ 수집된 데이터가 없습니다.")

# 10. 사용 가능한 변수들 안내
print(f"\n🔧 사용 가능한 변수들:")
print(f"• info: 수집된 최종 데이터프레임 ({len(info) if not info.empty else 0} rows)")
print(f"• successful_tickers: 성공한 티커 리스트 ({len(successful_tickers)}개)")
print(f"• failed_tickers: 실패한 티커 리스트 ({len(failed_tickers)}개)")
print(f"• ticker_list: 원본 처리 대상 티커 리스트 ({len(ticker_list)}개)")

print(f"\n📝 사용 예시:")
print(f"• 특정 티커 성공 여부: print('TSLA' in successful_tickers)")
print(f"• 실패 티커 확인: print(failed_tickers[:10])")
print(f"• DataFrame 정보: print(info.info())")

print(f"\n🎊 수집 작업이 완료되었습니다!")

🚀 ============================================================
📈 개선된 FMP API 주식 데이터 수집기 시작!
⚙️  동시 요청 수: 10
⏱️  기본 딜레이: 1.0초
🔄 최대 재시도: 5회
📦 배치 크기: 100
📋 주식 리스트 가져오는 중...
✅ 주식 리스트 수신 완료!
📊 데이터 필터링 중...
📊 필터링 완료! 처리 대상 티커: 11684개

⚠️  대용량 데이터입니다. 전체 처리 시 수 시간이 소요될 수 있습니다.
📝 테스트용으로 처음 500개만 처리하려면 'y'를 입력하세요.
🎯 테스트 모드: 500개 티커만 처리합니다.

🚀 데이터 수집 시작!
📊 총 500개 종목 처리 예정
🔄 기존 이벤트 루프에서 실행 중...

🎯 배치 1/5 시작 (100개 종목)
📡 HNI 요청 중... (시도 1/5)
📡 FAST 요청 중... (시도 1/5)
📡 GOOGL 요청 중... (시도 1/5)
📡 NVDA 요청 중... (시도 1/5)
✅ HNI 성공!
📡 BEKE 요청 중... (시도 1/5)
📡 INTC 요청 중... (시도 1/5)
📡 TRS 요청 중... (시도 1/5)
📡 LTBR 요청 중... (시도 1/5)
✅ BEKE 성공!
📡 RAL 요청 중... (시도 1/5)
✅ FAST 성공!
✅ GOOGL 성공!
✅ RAL 성공!
🔄 진행: [██░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 5/100 (5.0%)
✅ 성공: 5 | ❌ 실패: 0 | 📈 성공률: 100.0%
⏱️  경과: 0:16 | 예상잔여: 5:11
----------------------------------------------------------------------
📡 WRBY 요청 중... (시도 1/5)
✅ NVDA 성공!
✅ WRBY 성공!
✅ INTC 성공!
✅ TRS 성공!
✅ LTBR 성공!
🔄 진행: [████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 10

C:\Users\82108\AppData\Local\Temp\ipykernel_18976\3991531980.py:420: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  info = pd.concat([df for df in all_results if df is not None], ignore_index=True)


In [9]:
# import pandas as pd
# import json
# import ssl
# import certifi
# from urllib.request import urlopen
# from urllib.error import HTTPError
# import time
# import asyncio
# import aiohttp
# import logging
from datetime import datetime, date
from dateutil.relativedelta import relativedelta

# 로깅 설정 (ERROR만 표시)
logging.basicConfig(level=logging.ERROR)
logger = logging.getLogger(__name__)

# API 키 설정
api_key = "hT0gAk87j9xZx4PlBApvBqfVL5IahvgV"

# 날짜 관련 함수들 (기존 함수가 정의되어 있지 않은 경우를 위해)
def get_last_quarter_end():
    """마지막 분기 마지막 날을 반환하는 함수"""
    today = date.today()
    quarter_months = [3, 6, 9, 12]

    # 현재 월에서 가장 가까운 이전 분기 마지막 월 찾기
    current_month = today.month
    last_quarter_month = None

    for month in reversed(quarter_months):
        if month < current_month:
            last_quarter_month = month
            break

    # 만약 현재가 1-3월이라면 작년 12월
    if last_quarter_month is None:
        last_quarter_month = 12
        year = today.year - 1
    else:
        year = today.year

    # 해당 월의 마지막 날 구하기
    import calendar
    last_day = calendar.monthrange(year, last_quarter_month)[1]

    return date(year, last_quarter_month, last_day)

def get_last_month_end():
    """지난 달 마지막 날을 반환하는 함수"""
    today = date.today()
    # 이번 달 첫째 날에서 하루 빼기 = 지난 달 마지막 날
    first_day_this_month = today.replace(day=1)
    last_month_end = first_day_this_month - relativedelta(days=1)
    return last_month_end

# 초기 세팅 (기존 코드)
today_date = datetime.now().strftime("%Y-%m-%d")
last_quarter_end = get_last_quarter_end()
quarter_date = last_quarter_end.strftime("%Y-%m-%d")
last_month_date = get_last_month_end()

# info 데이터 필터링
info_df = info[['symbol', 'sector', 'industry']]

con1 = info_df['sector'] != 'Financial Services'
con2 = info_df['sector'] != 'Real Estate'
con3 = info_df['sector'] != 'Utilities'
con4 = info_df['sector'] != ''

info_re = info_df[con1 & con2 & con3 & con4].copy()
tic_list = info_re['symbol'].unique().tolist()

print(f"Total tickers to process: {len(tic_list)}")

# 컬럼 매핑
compustat_is = ['report_date', 'ticker', 'period', 'sale', 'cogs', 'gp', 'xrd', 'xsga', 'idit',
                'xint', 'dp', 'ebitda', 'xopr', 'opiti', 'opir', 'pi', 'pir', 'txt', 'ni',
                'nir', 'eps', 'epsdi', 'shrout', 'shroutdi']

# 날짜 범위 생성
dates_list = pd.date_range('2004-01-31', last_month_date, freq='M')
date_df = pd.DataFrame(dates_list, columns=['date'])

# 비동기 설정
max_concurrent = 25  # 재무제표는 더 신중하게
delay = 0.08  # 딜레이 증가

# 동기 함수 (fallback용)
def get_jsonparsed_data(url):
    context = ssl.create_default_context(cafile=certifi.where())
    with urlopen(url, context=context, timeout=15) as response:
        data = response.read().decode("utf-8")
        return json.loads(data)

# 비동기 함수들
async def fetch_json_data(url, session, semaphore, retry_count=3):
    """비동기로 JSON 데이터를 가져오는 함수"""
    async with semaphore:
        for attempt in range(retry_count):
            try:
                await asyncio.sleep(delay)
                async with session.get(url) as response:
                    if response.status == 200:
                        data = await response.text()
                        return json.loads(data)
                    elif response.status == 429:
                        wait_time = 2 ** attempt
                        await asyncio.sleep(wait_time)
                    else:
                        return None

            except asyncio.TimeoutError:
                if attempt == retry_count - 1:
                    return None

            except Exception as e:
                if attempt == retry_count - 1:
                    return None
                await asyncio.sleep(1)

        return None

async def fetch_income_statement(ticker, session, semaphore):
    """개별 티커의 손익계산서를 가져오고 처리하는 함수"""
    url = f"https://financialmodelingprep.com/api/v3/income-statement/{ticker}?period=quarter&apikey={api_key}"

    try:
        fs_raw = await fetch_json_data(url, session, semaphore)

        if not fs_raw:
            return None, ticker  # 실패한 티커 반환

        temp_df = pd.DataFrame(fs_raw)

        if temp_df.empty:
            return None, ticker

        # 컬럼 선택 및 이름 변경
        required_columns = ['date', 'symbol', 'period', 'revenue', 'costOfRevenue', 'grossProfit',
                          'researchAndDevelopmentExpenses', 'sellingGeneralAndAdministrativeExpenses',
                          'interestIncome', 'interestExpense', 'depreciationAndAmortization', 'ebitda',
                          'operatingExpenses', 'operatingIncome', 'operatingIncomeRatio', 'incomeBeforeTax',
                          'incomeBeforeTaxRatio', 'incomeTaxExpense', 'netIncome', 'netIncomeRatio',
                          'eps', 'epsdiluted', 'weightedAverageShsOut', 'weightedAverageShsOutDil']

        # 존재하는 컬럼만 선택
        available_columns = [col for col in required_columns if col in temp_df.columns]
        is_df = temp_df[available_columns].copy()

        # 컬럼명 매핑 (길이 맞춤)
        column_mapping = dict(zip(available_columns, compustat_is[:len(available_columns)]))
        is_df = is_df.rename(columns=column_mapping)

        # 날짜 처리
        is_df_sorted = is_df.sort_values(by='report_date')
        is_df_sorted['report_date'] = pd.to_datetime(is_df_sorted['report_date'])
        is_df_sorted['date_month'] = is_df_sorted['report_date'].dt.to_period('M').astype(str)
        is_df_sorted['date'] = pd.to_datetime(is_df_sorted['date_month']) + pd.offsets.MonthEnd(0)

        # 날짜 범위와 병합
        temp_is = pd.merge(date_df, is_df_sorted, on=['date'], how='left').ffill()

        return temp_is, None  # 성공

    except Exception as e:
        return None, ticker  # 실패한 티커 반환

async def download_all_income_statements(ticker_list):
    """모든 티커의 손익계산서를 비동기로 다운로드"""
    ssl_context = ssl.create_default_context(cafile=certifi.where())
    connector = aiohttp.TCPConnector(
        ssl=ssl_context,
        limit=60,
        limit_per_host=30
    )

    timeout = aiohttp.ClientTimeout(total=45)
    semaphore = asyncio.Semaphore(max_concurrent)

    async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:
        tasks = [fetch_income_statement(ticker, session, semaphore) for ticker in ticker_list]

        results = []
        error_list = []
        completed = 0
        successful = 0
        total = len(tasks)

        print(f"Starting download of {total} income statements...")

        for coro in asyncio.as_completed(tasks):
            try:
                result, failed_ticker = await coro
                if result is not None:
                    results.append(result)
                    successful += 1
                elif failed_ticker:
                    error_list.append(failed_ticker)

                completed += 1

                # 50개마다 진행률 출력
                if completed % 50 == 0 or completed == total:
                    print(f"Progress: {completed}/{total} ({completed/total*100:.1f}%) - Success: {successful} - Errors: {len(error_list)}")

            except Exception as e:
                completed += 1
                error_list.append("unknown")

    return results, error_list

# 실행 및 데이터 다운로드
start_time = time.time()

# 배치 처리 설정
batch_size = 500  # 재무제표는 더 작은 배치로
all_is_list = []
all_error_list = []

if len(tic_list) > batch_size:
    print(f"Processing in batches of {batch_size}")

    for i in range(0, len(tic_list), batch_size):
        batch = tic_list[i:i + batch_size]
        batch_num = i // batch_size + 1
        total_batches = (len(tic_list) + batch_size - 1) // batch_size

        print(f"\nProcessing batch {batch_num}/{total_batches}, size: {len(batch)}")

        try:
            batch_results, batch_errors = asyncio.run(download_all_income_statements(batch))
            all_is_list.extend(batch_results)
            all_error_list.extend(batch_errors)

        except Exception as e:
            print(f"Error processing batch {batch_num}: {e}")
            all_error_list.extend(batch)  # 전체 배치를 에러로 처리
            continue

        # 배치 간 대기 (서버 부하 방지)
        if i + batch_size < len(tic_list):
            print("Waiting 3 seconds before next batch...")
            time.sleep(3)

else:
    print("Processing all tickers in single batch...")
    try:
        all_is_list, all_error_list = asyncio.run(download_all_income_statements(tic_list))
    except Exception as e:
        print(f"Error in async processing: {e}")
        all_is_list = []
        all_error_list = tic_list

end_time = time.time()
print(f"\nDownload completed in {end_time - start_time:.2f} seconds")

# 결과 정리
if all_is_list:
    is_df = pd.concat([df for df in all_is_list if df is not None], ignore_index=True)
    print(f"Successfully processed {len(all_is_list)} tickers")
    print(f"Failed tickers: {len(all_error_list)}")
    print(f"Final DataFrame shape: {is_df.shape}")
else:
    is_df = pd.DataFrame()
    print("No data downloaded")

# 에러 리스트 출력 (처음 10개만)
if all_error_list:
    print(f"\nFirst 10 failed tickers: {all_error_list[:10]}")

# 결과 확인
if not is_df.empty:
    print("\nSample data:")
    print(is_df.head())
    print(f"\nDate range: {is_df['date'].min()} to {is_df['date'].max()}")
    print(f"Unique tickers: {is_df['ticker'].nunique()}")

print("\nProcessing complete!")

# 전역 변수로 결과 저장
error_list = all_error_list

Total tickers to process: 408
Processing all tickers in single batch...
Starting download of 408 income statements...


C:\Users\82108\AppData\Local\Temp\ipykernel_18976\1683451932.py:82: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  dates_list = pd.date_range('2004-01-31', last_month_date, freq='M')


Progress: 50/408 (12.3%) - Success: 49 - Errors: 1
Progress: 100/408 (24.5%) - Success: 98 - Errors: 2
Progress: 150/408 (36.8%) - Success: 147 - Errors: 3
Progress: 200/408 (49.0%) - Success: 197 - Errors: 3
Progress: 250/408 (61.3%) - Success: 246 - Errors: 4
Progress: 300/408 (73.5%) - Success: 296 - Errors: 4
Progress: 350/408 (85.8%) - Success: 296 - Errors: 54
Progress: 400/408 (98.0%) - Success: 296 - Errors: 104
Progress: 408/408 (100.0%) - Success: 296 - Errors: 112

Download completed in 46.50 seconds
Successfully processed 296 tickers
Failed tickers: 112
Final DataFrame shape: (76664, 26)

First 10 failed tickers: ['AIRO', 'GIBO', 'FLY', 'SDM', 'IDYA', 'VERI', 'AKBA', 'RXRX', 'ACB', 'INDI']

Sample data:
        date report_date ticker period  sale  cogs  gp  xrd  xsga  idit  ...  \
0 2004-01-31         NaT    NaN    NaN   NaN   NaN NaN  NaN   NaN   NaN  ...   
1 2004-02-29         NaT    NaN    NaN   NaN   NaN NaN  NaN   NaN   NaN  ...   
2 2004-03-31         NaT    NaN    

In [27]:
is_df

,date,report_date,ticker,period,sale,cogs,gp,xrd,xsga,idit,...,pi,pir,txt,ni,nir,eps,epsdi,shrout,shroutdi,date_month
5,2004-06-30,2004-06-30,UL,Q2,1.928300e+10,9.998146e+09,9.284854e+09,0.0,7.824572e+09,72500000.0,...,1.852000e+09,0.096043,405000000.0,1.377500e+09,0.071436,47.000000,45.000000,2.908709e+09,2.997297e+09,2004-06
6,2004-07-31,2004-06-30,UL,Q2,1.928300e+10,9.998146e+09,9.284854e+09,0.0,7.824572e+09,72500000.0,...,1.852000e+09,0.096043,405000000.0,1.377500e+09,0.071436,47.000000,45.000000,2.908709e+09,2.997297e+09,2004-06
7,2004-08-31,2004-06-30,UL,Q2,1.928300e+10,9.998146e+09,9.284854e+09,0.0,7.824572e+09,72500000.0,...,1.852000e+09,0.096043,405000000.0,1.377500e+09,0.071436,47.000000,45.000000,2.908709e+09,2.997297e+09,2004-06
8,2004-09-30,2004-06-30,UL,Q2,1.928300e+10,9.998146e+09,9.284854e+09,0.0,7.824572e+09,72500000.0,...,1.852000e+09,0.096043,405000000.0,1.377500e+09,0.071436,47.000000,45.000000,2.908709e+09,2.997297e+09,2004-06
9,2004-10-31,2004-06-30,UL,Q2,1.928300e+10,9.998146e+09,9.284854e+09,0.0,7.824572e+09,72500000.0,...,1.852000e+09,0.096043,405000000.0,1.377500e+09,0.071436,47.000000,45.000000,2.908709e+09,2.997297e+09,2004-06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76659,2025-03-31,2025-01-31,ADSK,Q4,1.633000e+09,1.660000e+08,1.467000e+09,393000000.0,6.910000e+08,3000000.0,...,3.720000e+08,0.227802,69000000.0,3.030000e+08,0.185548,1.409302,1.396313,2.150000e+08,2.170000e+08,2025-01
76660,2025-04-30,2025-04-30,ADSK,Q1,1.625000e+09,2.840000e+08,1.341000e+09,394000000.0,7.230000e+08,0.0,...,2.340000e+08,0.144000,82000000.0,1.520000e+08,0.093538,0.710000,0.700000,2.140000e+08,2.160000e+08,2025-04
76661,2025-05-31,2025-04-30,ADSK,Q1,1.625000e+09,2.840000e+08,1.341000e+09,394000000.0,7.230000e+08,0.0,...,2.340000e+08,0.144000,82000000.0,1.520000e+08,0.093538,0.710000,0.700000,2.140000e+08,2.160000e+08,2025-04
76662,2025-06-30,2025-04-30,ADSK,Q1,1.625000e+09,2.840000e+08,1.341000e+09,394000000.0,7.230000e+08,0.0,...,2.340000e+08,0.144000,82000000.0,1.520000e+08,0.093538,0.710000,0.700000,2.140000e+08,2.160000e+08,2025-04


In [28]:
import pandas as pd
import numpy as np
from itertools import product
from statsmodels.tsa.statespace.sarimax import SARIMAX
from tqdm import tqdm  # ✅ 진행상황 표시 라이브러리

# ================================
# SARIMA 예측 함수
# ================================
def sarima_forecast(df, date_col='date', value_col='sale', steps=12, use_log=True):
    """
    개별 시계열에 대해 SARIMA 기반 예측 수행
    """
    # 시계열 변환
    ts = df.set_index(date_col)[value_col].asfreq('Q')

    # 결측치 처리
    if ts.isna().any():
        ts = ts.interpolate()

    ts_transformed = np.log(ts) if use_log else ts

    # 파라미터 탐색
    p = d = q = P = D = Q = [0, 1]
    s = 4
    param_combinations = list(product(p, d, q))
    seasonal_combinations = list(product(P, D, Q))

    best_aic = np.inf
    best_model = None

    for (order, seasonal) in product(param_combinations, seasonal_combinations):
        try:
            model = SARIMAX(ts_transformed, order=order, seasonal_order=(*seasonal, s))
            result = model.fit(disp=False)
            if result.aic < best_aic:
                best_aic = result.aic
                best_model = result
        except:
            continue

    if best_model is None:
        return None

    # 예측
    forecast_log = best_model.forecast(steps=steps)
    forecast = np.exp(forecast_log) if use_log else forecast_log

    return pd.DataFrame({
        'date': pd.date_range(start=ts.index[-1] + pd.offsets.QuarterEnd(), periods=steps, freq='Q'),
        f'{value_col}_forecast': forecast.values
    })


# ================================
# 기업별 예측 함수 (진행상황 추가)
# ================================
def forecast_financials(df, ticker_col='ticker', date_col='date', value_col='sale', steps=12, min_obs=40):
    """
    전체 데이터프레임에서 기업별 SARIMA 예측 수행 (진행상황 표시 포함)
    """
    results = {}

    # 기업별 그룹핑
    groups = list(df.groupby(ticker_col))

    # tqdm 적용 (기업 개수만큼 진행 상황 표시)
    for ticker, sub_df in tqdm(groups, desc=f"예측 진행중 [{value_col}]", unit="기업"):
        if len(sub_df) < min_obs:
            continue  # 최소 데이터 개수 조건 미달

        forecast_df = sarima_forecast(
            sub_df[[date_col, value_col]].sort_values(date_col),
            date_col=date_col,
            value_col=value_col,
            steps=steps
        )

        if forecast_df is not None:
            results[ticker] = forecast_df

    return results


# ================================
# 사용 예시
# ================================
# is_df = ... (사용자의 데이터프레임)

# 매출(sale) 예측 (진행바와 ETA 표시됨)
sale_forecasts = forecast_financials(is_df, value_col='sale', steps=12)

# GP(매출총이익) 예측
gp_forecasts = forecast_financials(is_df, value_col='gp', steps=12)


예측 진행중 [sale]:   0%|          | 0/296 [00:00<?, ?기업/s]C:\Users\82108\AppData\Local\Temp\ipykernel_18976\2881298373.py:15: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  ts = df.set_index(date_col)[value_col].asfreq('Q')
C:\Users\82108\AppData\Local\Programs\Python\Python39\lib\site-packages\statsmodels\tsa\statespace\sarimax.py:1009: UserWarning: Non-invertible starting seasonal moving average Using zeros as starting parameters.
  warn('Non-invertible starting seasonal moving average'
C:\Users\82108\AppData\Local\Programs\Python\Python39\lib\site-packages\statsmodels\tsa\statespace\sarimax.py:997: UserWarning: Non-stationary starting seasonal autoregressive Using zeros as starting parameters.
  warn('Non-stationary starting seasonal autoregressive'
C:\Users\82108\AppData\Local\Programs\Python\Python39\lib\site-packages\statsmodels\tsa\statespace\sarimax.py:997: UserWarning: Non-stationary starting seasonal autoregressive Using zeros

In [29]:
sale_forecasts

{'AAPL':          date  sale_forecast
 0  2025-09-30   1.002976e+11
 1  2025-12-31   1.435201e+11
 2  2026-03-31   1.132396e+11
 3  2026-06-30   1.026302e+11
 4  2026-09-30   1.094641e+11
 5  2026-12-31   1.566368e+11
 6  2027-03-31   1.235889e+11
 7  2027-06-30   1.120099e+11
 8  2027-09-30   1.194683e+11
 9  2027-12-31   1.709523e+11
 10 2028-03-31   1.348840e+11
 11 2028-06-30   1.222468e+11,
 'ABCL':          date  sale_forecast
 0  2025-09-30   3.460686e+06
 1  2025-12-31   7.681186e+06
 2  2026-03-31   8.356619e+06
 3  2026-06-30   5.037669e+06
 4  2026-09-30   4.116598e+06
 5  2026-12-31   9.137018e+06
 6  2027-03-31   9.940467e+06
 7  2027-06-30   5.992469e+06
 8  2027-09-30   4.896826e+06
 9  2027-12-31   1.086878e+07
 10 2028-03-31   1.182451e+07
 11 2028-06-30   7.128235e+06,
 'ADSK':          date  sale_forecast
 0  2025-09-30   1.689674e+09
 1  2025-12-31   1.724794e+09
 2  2026-03-31   1.817152e+09
 3  2026-06-30   1.775679e+09
 4  2026-09-30   1.837982e+09
 5  2026-12-31

In [17]:
len(ticker_list)

500